# Creating Splits

In [1]:
# -------------------------
# 1️⃣ Importaciones y configuraciones
# -------------------------
import numpy as np
import json
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout, GRU, TimeDistributed
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

# -------------------------
# Cargar configuración desde config.json
# -------------------------
with open('../config.json', 'r') as f:
    config = json.load(f)

# Datos comunes
common_config = config['common']
ACTIONS = common_config['actions']
SEQUENCE_LENGTH = common_config['sequence_length']

# Datos específicos de train_model
train_config = config['train_model']
DATASET_PATH = train_config['dataset_path']
MODEL_EXPORT_NAME = train_config['model_export_name']
HAND_SELECTION = train_config['hand_selection']

# Configuración de arquitectura del modelo
model_arch = train_config['model_architecture']
MODEL_TYPE = model_arch['type']
LAYERS_CONFIG = model_arch['layers']

# Configuración de entrenamiento
training_config = train_config['training']
EPOCHS = training_config['epochs']
BATCH_SIZE = training_config['batch_size']
VALIDATION_SPLIT = training_config['validation_split']
OPTIMIZER = training_config['optimizer']
LOSS = training_config['loss']
METRICS = training_config['metrics']

# Configuración de split de datos
split_config = train_config['data_split']
TEST_SIZE = split_config['test_size']
STRATIFY = split_config['stratify']

# -------------------------
# 2️⃣ Cargar dataset desde archivo
# -------------------------
data = np.load(DATASET_PATH)
X = data['X']  # forma original: (num_samples, sequence_length, 21, 3)
y_labels = data['y']

# Seleccionar mano(s) según configuración
if HAND_SELECTION == 'left':
    # Solo mano izquierda: primeros 21 landmarks
    X = X[:, :, 22:, :]
    print(f"Usando solo mano izquierda: {X.shape}")
elif HAND_SELECTION == 'right':
    # Solo mano derecha: últimos 21 landmarks
    X = X[:, :, :22, :]
    print(f"Usando solo mano derecha: {X.shape}")
else:  # 'both'
    # Ambas manos: mantener todos los 42 landmarks
    print(f"Usando ambas manos: {X.shape}")

# Aplanar landmarks de cada frame: (21,3) -> (63) o (42,3) -> (126)
num_samples = X.shape[0]
X = X.reshape(num_samples, SEQUENCE_LENGTH, -1)  # ahora (num_samples, sequence_length, features)

# Convertir labels a one-hot
y = to_categorical(y_labels, num_classes=len(ACTIONS))

# -------------------------
# 3️⃣ Train / Test split
# -------------------------
stratify_param = y_labels if STRATIFY else None
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=stratify_param, random_state=42
)

print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("y_train:", y_train.shape, "y_test:", y_test.shape)

# -------------------------
# 4️⃣ Crear modelo
# -------------------------
model = Sequential()

# Construir capas según configuración
for i, layer_config in enumerate(LAYERS_CONFIG):
    layer_type = layer_config['type']
    
    if layer_type == 'TimeDistributed_Dense':
        units = layer_config['units']
        activation = layer_config['activation']
        if i == 0:
            # Primera capa necesita input_shape
            model.add(TimeDistributed(Dense(units, activation=activation), 
                                     input_shape=(SEQUENCE_LENGTH, X.shape[2])))
        else:
            model.add(TimeDistributed(Dense(units, activation=activation)))
    
    elif layer_type == 'Dropout':
        rate = layer_config['rate']
        model.add(Dropout(rate))
    
    elif layer_type == 'GRU':
        units = layer_config['units']
        return_sequences = layer_config.get('return_sequences', False)
        model.add(GRU(units, return_sequences=return_sequences))
    
    elif layer_type == 'Dense':
        units = layer_config.get('units', len(ACTIONS))
        activation = layer_config['activation']
        model.add(Dense(units, activation=activation))

model.compile(
    optimizer=OPTIMIZER,
    loss=LOSS,
    metrics=METRICS
)

model.summary()

# -------------------------
# 5️⃣ Entrenamiento
# -------------------------
history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    validation_split=VALIDATION_SPLIT,
    batch_size=BATCH_SIZE,
    verbose=1
)

# Guardar modelo entrenado
model.save(MODEL_EXPORT_NAME)
print(f"Modelo guardado en {MODEL_EXPORT_NAME}")

# -------------------------
# 6️⃣ Evaluación / Métricas
# -------------------------
model = load_model(MODEL_EXPORT_NAME)

y_pred = np.argmax(model.predict(X_test), axis=1)
y_true = np.argmax(y_test, axis=1)

acc = accuracy_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)

print(f"Accuracy en test set: {acc:.4f}")
print("Matriz de confusión:")
print(cm)


2026-04-02 16:51:50.195465: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-02 16:51:50.252791: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-02 16:51:51.732930: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/home/luis/Documents/projects/python/HandActionDetectionModel/venv/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


Usando solo mano derecha: (3310, 20, 22, 3)
X_train: (2813, 20, 66) X_test: (497, 20, 66)
y_train: (2813, 12) y_test: (497, 12)


/home/luis/Documents/projects/python/HandActionDetectionModel/venv/lib/python3.12/site-packages/keras/src/layers/core/wrapper.py:27: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
2026-04-02 16:51:53.411805: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed                │ (None, 20, 64)         │         4,288 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 20, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 20, 64)         │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 64)             │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 12)             │           396 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 56,684 (221.42 KB)

 Trainable params: 56,684 (221.42 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
282/282 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.6436 - loss: 1.0721 - val_accuracy: 0.9556 - val_loss: 0.1747
Epoch 2/15
282/282 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.9702 - loss: 0.1213 - val_accuracy: 1.0000 - val_loss: 0.0164
Epoch 3/15
282/282 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.9996 - loss: 0.0140 - val_accuracy: 1.0000 - val_loss: 0.0048
Epoch 4/15
282/282 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.9973 - loss: 0.0164 - val_accuracy: 0.9094 - val_loss: 0.3921
Epoch 5/15
282/282 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.9818 - loss: 0.0763 - val_accuracy: 1.0000 - val_loss: 0.0033
Epoch 6/15
282/282 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 1.0000 - loss: 0.0036 - val_accuracy: 1.0000 - val_loss: 0.0016
Epoch 7/15
282/282 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.9822 - loss: 0.0693 - val_accuracy: 1.0000 - val_loss: 0.0026
Epoch 8/15
282/282 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 1.0000 - loss: 0.0028 - val_accu

Modelo guardado en hand_gesture_model.h5
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step
Accuracy en test set: 1.0000
Matriz de confusión:
[[38  0  0  0  0  0  0  0  0  0  0  0]
 [ 0 38  0  0  0  0  0  0  0  0  0  0]
 [ 0  0 38  0  0  0  0  0  0  0  0  0]
 [ 0  0  0 39  0  0  0  0  0  0  0  0]
 [ 0  0  0  0 38  0  0  0  0  0  0  0]
 [ 0  0  0  0  0 38  0  0  0  0  0  0]
 [ 0  0  0  0  0  0 39  0  0  0  0  0]
 [ 0  0  0  0  0  0  0 39  0  0  0  0]
 [ 0  0  0  0  0  0  0  0 38  0  0  0]
 [ 0  0  0  0  0  0  0  0  0 38  0  0]
 [ 0  0  0  0  0  0  0  0  0  0 38  0]
 [ 0  0  0  0  0  0  0  0  0  0  0 76]]
